# 模块四：游客客流与财报/股价对比分析


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from pathlib import Path

from config import CLEANED_TRAJ_DIR, STOCK_FINANCE_DIR, FIGURES_DIR, OUTPUT_DIR

plt.rcParams["font.sans-serif"] = ["STZhongsong", "SimHei", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False

analysis_out_dir = OUTPUT_DIR / "季度人数和营收"
backtest_out_dir = OUTPUT_DIR / "客流和股价"
analysis_out_dir.mkdir(parents=True, exist_ok=True)
backtest_out_dir.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# 统一映射口径：底层分析名称 -> 可视化展示名称
LOCATION_MERGE_MAP = {
    "张家界武陵源风景名胜区": "武陵源-宝峰湖合并景区",
    "宝峰湖风景区": "武陵源-宝峰湖合并景区"
}
LOCATION_DISPLAY_MAP = {
    "峨眉山": "峨眉山风景区",
    "武陵源-宝峰湖合并景区": "张家界武陵源风景名胜区 & 宝峰湖风景区",
    "长白山": "长白山风景区",
    "黄山": "黄山风景区"
}

# 统一公司简称映射：先标准化简称，再做唯一映射
COMPANY_TO_LOCATION = {
    "黄山旅游": "黄山",
    "峨眉山A": "峨眉山",
    "长白山": "长白山",
    "张家界": "武陵源-宝峰湖合并景区"
}

def normalize_security_name(name):
    s = str(name).strip().replace(" ", "")
    s = s.lstrip("*")
    if s.startswith("ST"):
        s = s[2:]
    return s

def first_existing_path(candidates):
    for p in candidates:
        if Path(p).exists():
            return Path(p)
    raise FileNotFoundError(f"未找到可用文件，候选路径: {candidates}")

## 1. 年度客流序列
先读取去重后的每日客流数据，合并张家界武陵源风景名胜区和宝峰湖风景区，绘制 2016-2023 年度独立用户数序列图。

In [ ]:
uv_data_path = first_existing_path([
    CLEANED_TRAJ_DIR / "final_tourists_uv.parquet",
    CLEANED_TRAJ_DIR / "合并_按照用户-日期-aoi去重.parquet"
])
raw_uv_df = pd.read_parquet(uv_data_path).copy()
raw_uv_df["date"] = pd.to_datetime(raw_uv_df["date"])

if "location" not in raw_uv_df.columns or "uid" not in raw_uv_df.columns:
    raise ValueError("客流数据缺少 location 或 uid 字段")

raw_uv_df["location_clean"] = raw_uv_df["location"].astype(str).map(
    lambda x: x.split("_", 1)[1] if "_" in x else x
)
raw_uv_df["location_agg"] = raw_uv_df["location_clean"].replace(LOCATION_MERGE_MAP)

raw_uv_df = raw_uv_df[
    (raw_uv_df["date"].dt.year >= 2016) & (raw_uv_df["date"].dt.year <= 2023)
]
raw_uv_df["year"] = raw_uv_df["date"].dt.year
raw_uv_df["quarter"] = raw_uv_df["date"].dt.quarter

yearly_uv = raw_uv_df.groupby(["location_agg", "year"])["uid"].nunique().reset_index(name="unique_users")
yearly_plot_df = yearly_uv.pivot(index="year", columns="location_agg", values="unique_users").sort_index()
yearly_plot_df = yearly_plot_df.rename(columns=LOCATION_DISPLAY_MAP)

yearly_plot_df.head()

In [ ]:
cm = 1 / 2.54
fig, ax = plt.subplots(figsize=(14 * cm, 9 * cm))

plot_colors = ["#55AE95", "#FD7792", "#3F4D71", "#FFAC8E"]
yearly_plot_df.plot(
    kind="line",
    ax=ax,
    marker="o",
    markersize=5,
    linewidth=1.8,
    markeredgewidth=1,
    markerfacecolor="white",
    color=plot_colors[: len(yearly_plot_df.columns)],
    alpha=0.9
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(False)
ax.set_axisbelow(True)
ax.set_title("各景区年用户量序列", fontsize=24, pad=18, fontweight="bold")
ax.set_xlabel("年份", fontsize=16, labelpad=12)
ax.set_ylabel("用户量", fontsize=16, labelpad=12)
ax.set_xticks(yearly_plot_df.index)
ax.tick_params(axis="x", labelsize=11, rotation=0)
ax.tick_params(axis="y", labelsize=11)
ax.legend(loc="upper left", frameon=False, fontsize=10)

fig1_path = FIGURES_DIR / "fig1_各景区年独立用户数序列.png"
plt.tight_layout()
plt.savefig(fig1_path, dpi=500, pad_inches=0.05, bbox_inches="tight")
plt.show()
print(f"已保存: {fig1_path}")

In [ ]:
finance_data_path = first_existing_path([
    STOCK_FINANCE_DIR / "营收_4家公司.csv",
    STOCK_FINANCE_DIR / "营收_四家公司.csv"
])
revenue_df = pd.read_csv(finance_data_path).copy()
revenue_df["REPORT_DATE"] = pd.to_datetime(revenue_df["REPORT_DATE"])
revenue_df["year"] = revenue_df["REPORT_DATE"].dt.year
revenue_df["quarter"] = revenue_df["REPORT_DATE"].dt.quarter

# 先标准化简称，再做唯一映射，避免 ST/*ST/去帽 等前后缀导致重复口径
revenue_df["SECURITY_NAME_STD"] = revenue_df["SECURITY_NAME_ABBR"].map(normalize_security_name)
revenue_df["location"] = revenue_df["SECURITY_NAME_STD"].map(COMPANY_TO_LOCATION)

unmapped_rows = revenue_df.loc[
    revenue_df["location"].isna(),
    ["SECURITY_NAME_ABBR", "SECURITY_NAME_STD"]
]
unmapped_rows = unmapped_rows.drop_duplicates()
if not unmapped_rows.empty:
    print("警告：以下公司简称未映射，将在后续分析中被过滤：")
    print(unmapped_rows.to_string(index=False))

revenue_df = revenue_df[revenue_df["location"].notna()].copy()

quarterly_uv_long = raw_uv_df.groupby(["location_agg", "year", "quarter"])["uid"].nunique().reset_index(name="UNIQUE_USER_COUNT")
quarterly_uv_long = quarterly_uv_long.rename(columns={"location_agg": "location"})

quarterly_uv_wide = quarterly_uv_long.pivot_table(
    index=["year", "quarter"],
    columns="location",
    values="UNIQUE_USER_COUNT",
    aggfunc="sum"
).reset_index()
quarterly_uv_wide.to_csv(analysis_out_dir / "季度独立用户数.csv", index=False)

merged_full = revenue_df.merge(
    quarterly_uv_long,
    on=["year", "quarter", "location"],
    how="left"
).sort_values(["location", "year", "quarter"])

merged_full["visitors_yoy"] = merged_full.groupby("location")["UNIQUE_USER_COUNT"].pct_change(4)
merged_full["revenue_yoy"] = merged_full.groupby("location")["TOTAL_OPERATE_INCOME"].pct_change(4)
merged_full = merged_full.set_index(["year", "quarter"]).sort_index()

merged_full.to_csv(analysis_out_dir / "季度独立用户数和营收.csv", index=True)
merged_full[["location", "UNIQUE_USER_COUNT", "TOTAL_OPERATE_INCOME", "visitors_yoy", "revenue_yoy"]].head()

## 2. 同比相关性与滞后检验
对 2016-2023 年季度样本计算：
- 同期相关性 $R0$：当季客流同比 vs 当季营收同比
- 领先一期 $R1$：上季客流同比 vs 当季营收同比
- 领先二期 $R2$：上两季客流同比 vs 当季营收同比

In [ ]:
analysis_df = merged_full.reset_index().copy()
analysis_df = analysis_df[(analysis_df["year"] >= 2016) & (analysis_df["year"] <= 2023)].copy()
analysis_df = analysis_df.set_index(["year", "quarter"]).sort_index()

locations = ["峨眉山", "武陵源-宝峰湖合并景区", "长白山", "黄山"]
plot_name_map = {
    "峨眉山": "峨眉山A",
    "武陵源-宝峰湖合并景区": "张家界",
    "长白山": "长白山",
    "黄山": "黄山旅游"
}

def calc_lag_corr(df_loc, lag):
    temp = pd.concat([df_loc["visitors_yoy"].shift(lag), df_loc["revenue_yoy"]], axis=1).dropna()
    if len(temp) <= 4:
        return np.nan, np.nan
    return pearsonr(temp.iloc[:, 0], temp.iloc[:, 1])

cm = 1 / 2.54
fig, axes = plt.subplots(2, 2, figsize=(16 * cm, 16 * cm))
axes = axes.flatten()
lag_results = []

for i, loc in enumerate(locations):
    ax1 = axes[i]
    df_loc = analysis_df[analysis_df["location"] == loc].dropna(subset=["visitors_yoy", "revenue_yoy"]).copy()
    if df_loc.empty:
        continue

    r0, p0 = pearsonr(df_loc["visitors_yoy"], df_loc["revenue_yoy"])
    r1, p1 = calc_lag_corr(df_loc, 1)
    r2, p2 = calc_lag_corr(df_loc, 2)

    lag_results.append({
        "景区": plot_name_map.get(loc, loc),
        "同期(R0)": r0,
        "领先1期(R1)": r1,
        "领先2期(R2)": r2,
        "P值(同期)": p0
    })

    x_labels = [f"{int(y)} Q{int(q)}" for y, q in df_loc.index]
    x_ticks = np.arange(len(x_labels))

    l1 = ax1.plot(x_ticks, df_loc["visitors_yoy"], color="tab:blue", linewidth=1.5, label="客流同比增速")
    ax1.set_ylabel("景区客流同比增速", color="tab:blue", fontsize=9)
    ax1.tick_params(axis="y", labelcolor="tab:blue")

    ax2 = ax1.twinx()
    l2 = ax2.plot(
        x_ticks,
        df_loc["revenue_yoy"],
        color="tab:red",
        linestyle="--",
        linewidth=1.5,
        label="营收同比增速"
    )
    ax2.set_ylabel("企业营收同比增速", color="tab:red", fontsize=9)
    ax2.tick_params(axis="y", labelcolor="tab:red")

    ax1.axhline(0, color="black", linewidth=1, alpha=0.5)
    ax1.set_title(f"{plot_name_map.get(loc, loc)}\n(同期相关性 $r$ = {r0:.2f})", fontsize=10, fontweight="bold")
    ax1.set_xticks(x_ticks[::2])
    ax1.set_xticklabels(x_labels[::2], rotation=45, ha="right", rotation_mode="anchor", fontsize=8)

    handles = l1 + l2
    ax1.legend(handles, [h.get_label() for h in handles], loc="upper left", fontsize=8)

plt.tight_layout()
fig2_path = FIGURES_DIR / "fig2_各景区同比增速相关性分析.png"
plt.savefig(fig2_path, dpi=500, pad_inches=0.01, bbox_inches="tight")
plt.show()

lag_summary = pd.DataFrame(lag_results)
if not lag_summary.empty:
    lag_summary["最优预测期"] = lag_summary[["同期(R0)", "领先1期(R1)", "领先2期(R2)"]].idxmax(axis=1)
print(lag_summary)
print(f"已保存: {fig2_path}")

In [ ]:
lag_csv_path = analysis_out_dir / "同期和滞后相关性.csv"
lag_summary.to_csv(lag_csv_path, index=False)
print(f"已保存: {lag_csv_path}")

## 3. 客流因子多空回测
使用季度客流同比增速作为排序信号：每季度做多前2、做空后2，持有下一季度，统计多空收益差与累计净值。

In [ ]:
stock_data_path = first_existing_path([
    STOCK_FINANCE_DIR / "每日前复权价格_4只股票.csv"
])
stock_df = pd.read_csv(stock_data_path, dtype={"股票代码": str}).copy()

required_cols = {"日期", "股票代码", "收盘"}
missing_cols = required_cols - set(stock_df.columns)
if missing_cols:
    raise ValueError(f"股价数据缺少字段: {missing_cols}")

stock_df["日期"] = pd.to_datetime(stock_df["日期"])
stock_df["股票代码"] = stock_df["股票代码"].astype(str).str.zfill(6)
stock_df["year"] = stock_df["日期"].dt.year
stock_df["quarter"] = stock_df["日期"].dt.quarter

if "成交量" in stock_df.columns:
    stock_df["is_suspended"] = (stock_df["成交量"] == 0) | stock_df["成交量"].isna()
    critical_suspensions = stock_df.sort_values("日期").groupby(["股票代码", "year", "quarter"]).tail(1)
    critical_suspensions = critical_suspensions[critical_suspensions["is_suspended"]]
    if critical_suspensions.empty:
        print("季度末调仓日未检测到停牌。")
    else:
        print("警告：存在季度末停牌样本，请检查：")
        print(critical_suspensions[["日期", "股票代码", "成交量"]].head())

stock_df.head()

In [ ]:
location_to_code = {
    "峨眉山": "000888",
    "武陵源-宝峰湖合并景区": "000430",
    "长白山": "603099",
    "黄山": "600054"
}

quarterly_prices = stock_df.sort_values("日期").groupby(["股票代码", "year", "quarter"]).tail(1).copy()
quarterly_prices = quarterly_prices[["股票代码", "year", "quarter", "收盘"]].sort_values(["股票代码", "year", "quarter"]).copy()
quarterly_prices["next_q_return"] = quarterly_prices.groupby("股票代码")["收盘"].pct_change().shift(-1)

signal_df = merged_full.reset_index()[["year", "quarter", "location", "visitors_yoy"]].copy()
signal_df = signal_df[(signal_df["year"] >= 2017) & (signal_df["year"] <= 2023)]
signal_df["股票代码"] = signal_df["location"].map(location_to_code)

strategy_df = signal_df.merge(
    quarterly_prices[["股票代码", "year", "quarter", "next_q_return"]],
    on=["股票代码", "year", "quarter"],
    how="inner"
).dropna(subset=["visitors_yoy", "next_q_return"])

valid_quarters = strategy_df.groupby(["year", "quarter"])["股票代码"].nunique()
valid_quarters = valid_quarters[valid_quarters >= 4].index
strategy_df = strategy_df.set_index(["year", "quarter"]).loc[valid_quarters].reset_index()

strategy_df.head()

In [ ]:
def calc_long_short_spread(group):
    ranked = group.sort_values("visitors_yoy", ascending=False)
    long_ret = ranked.head(2)["next_q_return"].mean()
    short_ret = ranked.tail(2)["next_q_return"].mean()
    return long_ret - short_ret

backtest_results = strategy_df.groupby(["year", "quarter"]).apply(calc_long_short_spread).rename("spread")
nav_df = backtest_results.to_frame().copy()
nav_df["cum_net_value"] = (1 + nav_df["spread"]).cumprod()

quarterly_spread = nav_df.reset_index()
diag_df = strategy_df[strategy_df["year"] >= 2022].copy()
diag_df["rank"] = diag_df.groupby(["year", "quarter"])["visitors_yoy"].rank(ascending=False)

print("--- 2022Q4-2023Q4 季度多空收益明细 ---")
print(quarterly_spread[(quarterly_spread["year"] > 2022) | ((quarterly_spread["year"] == 2022) & (quarterly_spread["quarter"] >= 4))].to_string(index=False))

print("\n--- 2023年个股信号与收益对比 ---")
print(diag_df[diag_df["year"] == 2023].sort_values(["year", "quarter", "rank"]).to_string(index=False))

In [ ]:
def compute_metrics(df):
    if df.empty:
        return None

    work = df.copy()
    work["cum_net_value"] = (1 + work["spread"]).cumprod()
    n_quarters = len(work)
    terminal_nav = work["cum_net_value"].iloc[-1]
    cumulative_return = terminal_nav - 1
    win_rate = (work["spread"] > 0).mean()
    avg_spread = work["spread"].mean()

    peak = work["cum_net_value"].cummax()
    drawdown = (work["cum_net_value"] - peak) / peak
    max_drawdown = drawdown.min()

    cagr = terminal_nav ** (4 / n_quarters) - 1 if n_quarters > 0 else np.nan
    annual_return = avg_spread * 4
    annual_vol = work["spread"].std() * np.sqrt(4)
    sharpe = annual_return / annual_vol if pd.notna(annual_vol) and annual_vol != 0 else np.nan
    calmar = cagr / abs(max_drawdown) if pd.notna(max_drawdown) and max_drawdown != 0 else np.nan

    return {
        "胜率": f"{win_rate:.2%}",
        "平均季度多空收益差": f"{avg_spread:.2%}",
        "累计多空收益": f"{cumulative_return:.2%}",
        "CAGR": f"{cagr:.2%}",
        "最大回撤": f"{max_drawdown:.2%}",
        "卡玛比率": f"{calmar:.2f}",
        "夏普比率": f"{sharpe:.2f}"
    }

result_full = nav_df.reset_index()
normal_mask = (result_full["year"] < 2022) | ((result_full["year"] == 2022) & (result_full["quarter"] <= 3))
normal_period_df = result_full[normal_mask]

performance_table = pd.DataFrame({
    "2017Q1-2022Q3（正常期）": compute_metrics(normal_period_df),
    "2017Q1-2023Q4（完整期）": compute_metrics(result_full)
}).T.reset_index().rename(columns={"index": "时期"})

metrics_path = backtest_out_dir / "策略表现指标_2017-2022和2017-2023.csv"
performance_table.to_csv(metrics_path, index=False)
display(performance_table)
print(f"已保存: {metrics_path}")

In [ ]:
cm = 1 / 2.54
fig, ax = plt.subplots(figsize=(16 * cm, 9 * cm))

plot_df = nav_df.reset_index().copy()
x = np.arange(len(plot_df))
x_labels = [f"{int(y)} Q{int(q)}" for y, q in zip(plot_df["year"], plot_df["quarter"])]

ax.plot(x, plot_df["cum_net_value"], color="#134E8E", marker="o", markersize=4.5, linewidth=1.8)
ax.axhline(1.0, color="grey", linestyle="--", alpha=0.5)

split_candidates = plot_df[(plot_df["year"] == 2023) & (plot_df["quarter"] == 1)].index.tolist()
if split_candidates:
    split_idx = split_candidates[0]
    vline_pos = split_idx - 0.5
    ax.axvline(x=vline_pos, color="#A82323", linestyle="--", linewidth=1.8)
    ax.axvspan(-0.5, vline_pos, facecolor="#66D0BC", alpha=0.18)
    ax.axvspan(vline_pos, len(plot_df) - 0.5, facecolor="#FF88BA", alpha=0.20)
    ax.text(len(plot_df) * 0.22, plot_df["cum_net_value"].max() * 0.95, "正常期", fontsize=10,
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "black", "linewidth": 1})
    ax.text(len(plot_df) * 0.88, plot_df["cum_net_value"].max() * 0.90, "低基数\n失真期", fontsize=10,
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "black", "linewidth": 1})

ax.set_title("客流因子多空组合累计净值", fontsize=13, fontweight="bold")
ax.set_ylabel("累计净值", fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
ax.grid(True, linestyle=":", alpha=0.6)

fig3_path = FIGURES_DIR / "fig3_客流因子多空组合累计净值.png"
plt.tight_layout()
plt.savefig(fig3_path, dpi=500, pad_inches=0.01, bbox_inches="tight")
plt.show()
print(f"已保存: {fig3_path}")